Dada a base CIFAR10 que contém imagens organizadas em 10 classes, considere a criação de um classificador.

Estratégia a serem avaliadas:

3) ajuste fino (fine tuning) de uma CNN pré-treinada na base de imagens ImageNet

In [2]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Flatten
from tensorflow.keras.applications import VGG16
from tensorflow.keras import datasets, layers, models
from tensorflow.keras.utils import to_categorical

Dispositivos encontrados: [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU'), PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [3]:
# 1. Carregando e preparando os dados CIFAR-10
(train_images, train_labels), (test_images, test_labels) = datasets.cifar10.load_data()
train_images = train_images.astype('float32') / 255.0
test_images = test_images.astype('float32') / 255.0
train_labels = to_categorical(train_labels, 10)
test_labels = to_categorical(test_labels, 10)

In [4]:
# 2. Carregando a VGG16 pré-treinada
# include_top=False exclui as camadas densas originais (ImageNet)
vgg_conv = VGG16(weights='imagenet', include_top=False, input_shape=(32, 32, 3))

# Congelando as camadas para não destruir o aprendizado prévio do ImageNet
vgg_conv.trainable = False

I0000 00:00:1782622426.451046   43769 gpu_device.cc:2042] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 3606 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 2060, pci bus id: 0000:01:00.0, compute capability: 7.5


58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step


In [5]:
# 3. Criando o modelo com a base da VGG16 + nosso classificador customizado
model = Sequential()
model.add(vgg_conv)
model.add(Flatten())
model.add(Dense(2048, activation='relu'))
model.add(Dropout(0.3))
model.add(Dense(1024, activation='relu'))
model.add(Dense(10, activation='softmax'))

# 4. Compilando
model.compile(loss='categorical_crossentropy', 
              optimizer=tf.keras.optimizers.Adam(), 
              metrics=['accuracy'])

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ vgg16 (Functional)              │ (None, 1, 1, 512)      │    14,714,688 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 2048)           │     1,050,624 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1024)           │     2,098,176 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 10)             │        10,250 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 17,873,738 (68.18 MB)

 Trainable params: 3,159,050 (12.05 MB)

 Non-trainable params: 14,714,688 (56.13 MB)

In [6]:
# 5. Treinamento
# Com sua RTX 2060, este treinamento será muito eficiente
history = model.fit(train_images, train_labels, 
                    epochs=20, 
                    batch_size=64, 
                    validation_split=0.2, 
                    verbose=1)

# 6. Avaliação final
test_loss, test_acc = model.evaluate(test_images, test_labels)
print(f"\nAcurácia final com Fine-Tuning: {test_acc*100:.2f}%")

W0000 00:00:1782622545.566431   43769 cpu_allocator_impl.cc:82] Allocation of 491520000 exceeds 10% of free system memory.
W0000 00:00:1782622548.856063   43769 cpu_allocator_impl.cc:82] Allocation of 491520000 exceeds 10% of free system memory.


Epoch 1/20


I0000 00:00:1782622550.750359   48831 service.cc:154] XLA service 0x7c7b285163b0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1782622550.750428   48831 service.cc:170]   StreamExecutor [0]: NVIDIA GeForce RTX 2060, Compute Capability 7.5 (Driver: 13.0.0[580.97.0]; Runtime: 12.9.0; Toolkit: 12.5.0; DNN: 9.23.2)
I0000 00:00:1782622550.855455   48831 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1782622551.302576   48831 cuda_dnn.cc:436] Loaded cuDNN version 92302


 12/625 ━━━━━━━━━━━━━━━━━━━━ 6s 10ms/step - accuracy: 0.1979 - loss: 2.2633

I0000 00:00:1782622555.196689   48831 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


623/625 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.5098 - loss: 1.3877

W0000 00:00:1782622562.111886   43769 cpu_allocator_impl.cc:82] Allocation of 122880000 exceeds 10% of free system memory.
W0000 00:00:1782622562.353422   43769 cpu_allocator_impl.cc:82] Allocation of 122880000 exceeds 10% of free system memory.


625/625 ━━━━━━━━━━━━━━━━━━━━ 16s 17ms/step - accuracy: 0.5100 - loss: 1.3873 - val_accuracy: 0.5669 - val_loss: 1.2507
Epoch 2/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 12s 19ms/step - accuracy: 0.5764 - loss: 1.2022 - val_accuracy: 0.5886 - val_loss: 1.1886
Epoch 3/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 8s 12ms/step - accuracy: 0.6033 - loss: 1.1276 - val_accuracy: 0.5991 - val_loss: 1.1675
Epoch 4/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 7s 12ms/step - accuracy: 0.6192 - loss: 1.0759 - val_accuracy: 0.6083 - val_loss: 1.1479
Epoch 5/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 7s 12ms/step - accuracy: 0.6385 - loss: 1.0211 - val_accuracy: 0.6064 - val_loss: 1.1561
Epoch 6/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 8s 12ms/step - accuracy: 0.6512 - loss: 0.9800 - val_accuracy: 0.6095 - val_loss: 1.1691
Epoch 7/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 12s 18ms/step - accuracy: 0.6708 - loss: 0.9260 - val_accuracy: 0.6120 - val_loss: 1.1720
Epoch 8/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 8s 13ms/step - accuracy: 0.6854 - loss: 0.8824 - val_accuracy: 0.

W0000 00:00:1782622733.844044   43769 cpu_allocator_impl.cc:82] Allocation of 122880000 exceeds 10% of free system memory.


313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.6020 - loss: 1.5562

Acurácia final com Fine-Tuning: 60.20%
